In [ ]:
import torch
from datasets import load_dataset
from torch import nn
from transformers import Gemma4AudioFeatureExtractor, Gemma4AudioModel, Gemma4Model
from transformers.models.gemma4.modeling_gemma4 import (
    Gemma4AudioSubSampleConvProjectionLayer,
)
from transformers.monkey_patching import register_patch_mapping

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")

In [ ]:
feature_extractor = Gemma4AudioFeatureExtractor.from_pretrained("google/gemma-4-E2B")

In [ ]:
class FullTemporalGemma4AudioSubSampleConvProjectionLayer(
    Gemma4AudioSubSampleConvProjectionLayer
):
    def __init__(self, in_channels, out_channels, norm_eps):
        super().__init__(in_channels, out_channels, norm_eps)
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 3),
            stride=(1, 2),
            padding=1,
            bias=False,
        )

    @classmethod
    def register_patch_mapping(cls):
        register_patch_mapping(
            {Gemma4AudioSubSampleConvProjectionLayer.__name__: cls},
            overwrite=True,
        )


FullTemporalGemma4AudioSubSampleConvProjectionLayer.register_patch_mapping()
audio_encoder = Gemma4AudioModel.from_pretrained("google/gemma-4-E2B")

In [ ]:
audio_encoder

In [ ]:
audio = dataset[0]["audio"]

inputs = feature_extractor(
    [audio["array"]],
    sampling_rate=[audio["sampling_rate"]],
    return_tensors="pt",
)

In [ ]:
inputs

In [ ]:
with torch.inference_mode():
    outputs = audio_encoder(**inputs)

In [ ]:
outputs

In [ ]:
model = Gemma4Model.from_pretrained("google/gemma-4-E2B")
model